In [ ]:
# Path configuration
from pathlib import Path

PROJECT_ROOT = Path("..")  # Repository root.
BASELINE = PROJECT_ROOT / "data"
FIRM_DATA = PROJECT_ROOT
PATSTAT = PROJECT_ROOT / "patstat"
PATENT_TEXT = PROJECT_ROOT / "patent_text"
SAMPLEDATA = PROJECT_ROOT / "sampledata"
DATA = Path("data")          # Generated revision intermediates.
OUTPUTS = Path("outputs")    # Tables, figures and saved comparison results.
SEED = 42


# Robustness tables and figures

Run after notebooks 05–08. Read their saved comparison tables, cluster assignments and correspondence matrices, then draw the SI figures. Set `BASELINE`, `DATA` and `OUTPUTS` in the first cell. Save figures under `OUTPUTS/robustness/`.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from gbb_utils import read_profile, nonnoise, draw_correspondence

OUT = OUTPUTS / "robustness"
OUT.mkdir(parents=True, exist_ok=True)
original = read_profile(BASELINE / "clusangle_outlier_hdbscan2.parquet")
blocks = sorted(nonnoise(original).clus.unique())
EMBEDDINGS = [("paecter", "PaECTER"), ("patembed_base", "PatEmbed")]
PARAMETERS = [
    ("umap_neighbors_5", "UMAP neighbors: 5"),
    ("umap_neighbors_10", "UMAP neighbors: 10"),
    ("umap_neighbors_30", "UMAP neighbors: 30"),
    ("umap_neighbors_50", "UMAP neighbors: 50"),
    ("umap_components_2", "UMAP dimensions: 2"),
    ("umap_components_10", "UMAP dimensions: 10"),
    ("hdbscan_min_cluster_size_50", "Minimum cluster size: 50"),
    ("hdbscan_min_cluster_size_75", "Minimum cluster size: 75"),
    ("hdbscan_min_cluster_size_150", "Minimum cluster size: 150"),
    ("hdbscan_min_cluster_size_200", "Minimum cluster size: 200"),
    ("iqr_multiplier_1p0", "IQR multiplier: 1.0"),
    ("iqr_multiplier_1p25", "IQR multiplier: 1.25"),
    ("iqr_multiplier_1p75", "IQR multiplier: 1.75"),
    ("iqr_multiplier_2p0", "IQR multiplier: 2.0"),
    ("n_matches_5", "Retained matches: 5"),
    ("n_matches_8", "Retained matches: 8"),
    ("n_matches_15", "Retained matches: 15"),
    ("n_matches_20", "Retained matches: 20"),
]

## Read the analysis tables

Read the statistics and permutation benchmarks saved by notebooks 05–08.


In [ ]:
tables = [
    ("Embedding comparison", "embedding/embedding_comparison.csv"),
    ("GBB parameter settings", "gbb_parameters/stability.csv"),
    ("Source-field settings", "source_parameters/stability.csv"),
    ("CPC main groups", "main_groups/comparison.csv"),
    ("Detailed-code k-means", "alternatives/kmeans_nmi.csv"),
    ("Continuous method", "alternatives/continuous_nmi.csv"),
]
for title, filename in tables:
    print(title)
    display(pd.read_csv(OUTPUTS / filename))

## Alternative text embeddings

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5), constrained_layout=True)
for (model, title), ax in zip(EMBEDDINGS, axes):
    cells = pd.read_csv(OUTPUTS / "embedding" / f"{model}_correspondence.csv")
    partition = read_profile(DATA / "embeddings" / model / "gbbs.parquet")
    picture = draw_correspondence(cells, blocks, sorted(nonnoise(partition).clus.unique()), ax, title)
fig.colorbar(picture, ax=axes.tolist(), shrink=.7, label="RCA / (1 + RCA)")
fig.savefig(OUT / "embedding_correspondence.pdf")
fig.savefig(OUT / "embedding_correspondence.png", dpi=220)
plt.show()

## Eighteen changed GBB settings

In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(19.2, 9.9), constrained_layout=True)
for (variant, title), ax in zip(PARAMETERS, axes.ravel()):
    cells = pd.read_csv(OUTPUTS / "gbb_parameters" / f"{variant}_correspondence.csv")
    partition = read_profile(OUTPUTS / "gbb_parameters" / f"{variant}_partition.parquet")
    picture = draw_correspondence(cells, blocks, sorted(nonnoise(partition).clus.unique()), ax, title)
fig.colorbar(picture, ax=axes.ravel().tolist(), shrink=.7, label="RCA / (1 + RCA)")
fig.savefig(OUT / "parameter_correspondence.pdf")
fig.savefig(OUT / "parameter_correspondence.png", dpi=220)
plt.show()

## CPC main groups

Show the reference GBBs and main-group blocks with a reference counterpart. The saved correspondence table also records weights for codes without a corresponding cluster.


In [ ]:
cells = pd.read_csv(OUTPUTS / "main_groups/hierarchy_correspondence.csv")
cells = cells.loc[cells.clus_reference != "__unmapped__"]
fig, ax = plt.subplots(figsize=(5, 5), constrained_layout=True)
picture = draw_correspondence(cells, blocks, sorted(cells.clus_alternative.unique()), ax, "CPC main groups")
fig.colorbar(picture, ax=ax, shrink=.7, label="RCA / (1 + RCA)")
fig.savefig(OUT / "main_group_correspondence.pdf")
fig.savefig(OUT / "main_group_correspondence.png", dpi=220)
plt.show()

## Detailed-code k-means

Use the code-count correspondence for the figure. Notebook 08 separately saves the strength-weighted numerical NMI and its permutation benchmark.

In [ ]:
cells = pd.read_csv(OUTPUTS / "alternatives/kmeans_count_correspondence.csv")
cells = cells.rename(columns={"clus": "clus_alternative", "clus2": "clus_reference", "count": "score"})
fig, ax = plt.subplots(figsize=(5, 5), constrained_layout=True)
picture = draw_correspondence(cells, sorted(cells.clus_reference.unique()),
                             sorted(cells.clus_alternative.unique()), ax)
ax.set(xlabel="HDBSCAN", ylabel="K-means")
fig.colorbar(picture, ax=ax, shrink=.7, label="RCA / (1 + RCA)")
fig.savefig(OUT / "kmeans_correspondence.pdf")
fig.savefig(OUT / "kmeans_correspondence.png", dpi=220)
plt.show()

## Continuous orthogonal method

In [ ]:
cells = pd.read_csv(OUTPUTS / "alternatives/continuous_correspondence.csv")
cells = cells.rename(columns={"clus_reference": "clus_alternative", "clus_alternative": "clus_reference"})
fig, ax = plt.subplots(figsize=(5, 5), constrained_layout=True)
picture = draw_correspondence(cells, sorted(cells.clus_reference.unique()),
                             sorted(cells.clus_alternative.unique()), ax)
ax.set(xlabel="Original GBBs", ylabel="Continuous GBBs")
fig.colorbar(picture, ax=ax, shrink=.7, label="RCA / (1 + RCA)")
fig.savefig(OUT / "continuous_correspondence.pdf")
fig.savefig(OUT / "continuous_correspondence.png", dpi=220)
plt.show()